# AgriMarket: Machine Learning Model Evaluation & Error Metrics
**Note**: *This notebook is strictly for offline model evaluation, error calculations, and presentation. It is NOT part of the runtime web UI pipeline.*

### Topics & Techniques Covered:
1. **Introduction to Spark MLlib & PySpark Machine Learning Concepts**
2. **Regression Models & Error Evaluation**:
   - Multiple Linear Regression ($y = \beta_0 + \beta_1 X_1 + \beta_2 X_2$)
   - Intercept ($\beta_0$) and Coefficients ($\beta_1, \beta_2$)
   - Performance Metrics: **MAE (/qtl)**, **RMSE (/qtl)**, **MAPE (%)**, **$R^2$ Score**
3. **Classification Models & Performance Evaluation**:
   - Logistic Regression, Decision Tree Classifier, Linear SVM, Naive Bayes
   - Performance Matrix: **Confusion Matrix**, Precision, Recall, F1-Score
4. **Unsupervised Clustering**:
   - K-Means Market Profiling ($k=3$)
5. **Exploratory Correlation Analysis**:
   - Pearson Correlation Matrix and Feature Heatmap

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
print('Libraries loaded successfully.')

## 1. Load and Inspect Agrimarket Dataset
Dataset source: `data_engine/Dataset.csv` containing APMC mandi price records.

In [2]:
data_path = Path('../data_engine/Dataset.csv')
if not data_path.exists():
    data_path = Path('data_engine/Dataset.csv')

df = pd.read_csv(data_path)
print(f'Raw dataset shape: {df.shape}')
df.head()

## 2. Data Cleaning & Feature Preparation
Sanitize numeric columns: `Min_x0020_Price`, `Max_x0020_Price`, `Modal_x0020_Price`.

In [3]:
for col in ['Min_x0020_Price', 'Max_x0020_Price', 'Modal_x0020_Price']:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', ''), errors='coerce')

clean_df = df.dropna(subset=['Min_x0020_Price', 'Max_x0020_Price', 'Modal_x0020_Price']).copy()
clean_df = clean_df[(clean_df['Min_x0020_Price'] > 0) & (clean_df['Max_x0020_Price'] > 0) & (clean_df['Modal_x0020_Price'] > 0)]
print(f'Cleaned dataset records: {len(clean_df):,}')

## 3. Linear Regression: Coefficients, Intercept & Error Metrics
Predicting `Modal_x0020_Price` from `Min_x0020_Price` and `Max_x0020_Price`.
Mathematical Model:
$$\text{Modal Price} = \beta_0 + \beta_1 \cdot (\text{Min Price}) + \beta_2 \cdot (\text{Max Price}) + \epsilon$$

Target Performance Metrics:
- **MAE (/qtl)**: $\frac{1}{N} \sum |y - \hat{y}|$
- **RMSE (/qtl)**: $\sqrt{\frac{1}{N} \sum (y - \hat{y})^2}$
- **MAPE (%)**: $\frac{100\%}{N} \sum |\frac{y - \hat{y}}{y}|$
- **$R^2$ Score**: $1 - \frac{\sum (y - \hat{y})^2}{\sum (y - \bar{y})^2}$

In [4]:
X = clean_df[['Min_x0020_Price', 'Max_x0020_Price']]
y = clean_df['Modal_x0020_Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

reg = LinearRegression()
reg.fit(X_train, y_train)
y_pred = reg.predict(X_test)

# Error calculations
intercept = reg.intercept_
coef_min, coef_max = reg.coef_[0], reg.coef_[1]
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
r2 = r2_score(y_test, y_pred)

print('=' * 55)
print(' LINEAR REGRESSION PARAMETERS & EVALUATION METRICS')
print('=' * 55)
print(f'Intercept (beta_0):       {intercept:.4f}')
print(f'Coef Min Price (beta_1):  {coef_min:.4f}')
print(f'Coef Max Price (beta_2):  {coef_max:.4f}')
print(f'MAE (/qtl):               Rs. {mae:.2f} / quintal')
print(f'RMSE (/qtl):              Rs. {rmse:.2f} / quintal')
print(f'MAPE (%):                 {mape:.2f} %')
print(f'R2 Score:                 {r2:.4f}')
print('=' * 55)
print(f'Regression Formula: Modal = {intercept:.4f} + {coef_min:.4f}*(Min) + {coef_max:.4f}*(Max)')

### Residual Error Distribution Plot

In [5]:
residuals = y_test - y_pred
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, y_pred, alpha=0.3, color='#1b4332', s=10)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual Modal Price (Rs./qtl)')
axes[0].set_ylabel('Predicted Modal Price (Rs./qtl)')
axes[0].set_title(f'Actual vs Predicted (R2 = {r2:.4f})')

sns.histplot(residuals[(residuals > -2000) & (residuals < 2000)], kde=True, ax=axes[1], color='#2d6a4f')
axes[1].set_xlabel('Residual Error (Actual - Predicted)')
axes[1].set_title(f'Error Distribution (MAE: Rs.{mae:.2f}/qtl)')
plt.tight_layout()
plt.show()

## 4. Classification Models: Confusion Matrix & Performance Matrix
Classifying mandi price tiers (`Low`, `Medium`, `High`) based on price spread.
Models compared:
1. **Logistic Regression**
2. **Decision Tree Classifier**
3. **Linear Support Vector Machine (LinearSVC)**
4. **Gaussian Naive Bayes**

In [6]:
clean_df['Price_Tier'] = pd.qcut(clean_df['Modal_x0020_Price'], q=3, labels=['Low', 'Medium', 'High'])
clean_df['Spread'] = clean_df['Max_x0020_Price'] - clean_df['Min_x0020_Price']

X_cls = clean_df[['Min_x0020_Price', 'Max_x0020_Price', 'Spread']]
y_cls = clean_df['Price_Tier']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_c)
X_test_scaled = scaler.transform(X_test_c)

classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Linear SVM': LinearSVC(dual='auto', max_iter=2000, random_state=42),
    'Gaussian Naive Bayes': GaussianNB(),
}

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
axes = axes.flatten()
classes = ['Low', 'Medium', 'High']

for i, (name, clf) in enumerate(classifiers.items()):
    if name in ['Logistic Regression', 'Linear SVM']:
        clf.fit(X_train_scaled, y_train_c)
        preds = clf.predict(X_test_scaled)
    else:
        clf.fit(X_train_c, y_train_c)
        preds = clf.predict(X_test_c)
        
    cm = confusion_matrix(y_test_c, preds, labels=classes)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=classes, yticklabels=classes, ax=axes[i])
    axes[i].set_title(f'{name} - Confusion Matrix')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')
    
    print(f'=== Classification Report: {name} ===')
    print(classification_report(y_test_c, preds, labels=classes))

plt.tight_layout()
plt.show()

## 5. Unsupervised Learning: K-Means Clustering
Segmenting mandi price-arrival profiles into clusters.

In [7]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
sample_data = clean_df.sample(n=min(5000, len(clean_df)), random_state=42)[['Min_x0020_Price', 'Max_x0020_Price']]
sample_clusters = kmeans.fit_predict(sample_data)

plt.figure(figsize=(9, 6))
scatter = plt.scatter(sample_data['Min_x0020_Price'], sample_data['Max_x0020_Price'], c=sample_clusters, cmap='viridis', alpha=0.6, s=20)
centers = kmeans.cluster_centers_
plt.scatter(centers[:, 0], centers[:, 1], c='red', s=150, marker='X', label='Cluster Centroids')
plt.xlabel('Min Price (Rs./qtl)')
plt.ylabel('Max Price (Rs./qtl)')
plt.title('K-Means Clustering on Mandi Price Profiles (k=3)')
plt.legend()
plt.show()

## 6. Correlation Heatmap
Evaluating Pearson correlation across numerical market features.

In [8]:
corr_matrix = clean_df[['Min_x0020_Price', 'Max_x0020_Price', 'Modal_x0020_Price', 'Spread']].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt='.3f', linewidths=1)
plt.title('Pearson Correlation Matrix of Mandi Price Features')
plt.show()